# S07 — Object-Oriented Programming

**Python for Applied Physics** | Master in Applied Physics  
**Prerequisites:** S01–S06

---

## Learning objectives

By the end of this session you will be able to:

- Define classes with `__init__`, instance methods, and `__repr__`
- Use `@property` and setters for validated attribute access
- Implement special (dunder) methods to make objects behave like built-in types
- Build a class hierarchy with inheritance and `super()`
- Use `@dataclass` and `NamedTuple` for lightweight data containers
- Create parallel class-based implementations alongside existing functional code

---

## Session map

| Block | Topic | Cells |
|-------|-------|-------|
| 1 | Why OOP? | 1–3 |
| 2 | Classes and instances | 4–10 |
| 3 | Properties and validation | 11–14 |
| 4 | Special methods (dunder) | 15–18 |
| 5 | Inheritance | 19–23 |
| 6 | Dataclasses and NamedTuple | 24–27 |
| 7 | Integrating with `shared/` | 28–30 |

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import NamedTuple

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
print(f"NumPy {np.__version__}")

NumPy 2.4.3


---
## Block 1 — Why OOP?

Consider how we have been working so far: a pulse is represented as a loose collection of variables and the functions that operate on them are defined separately.

```python
# Procedural style — fragile
τ     = 100e-15
λ0    = 800e-9
energy = 50e-6

I = pulse_intensity(t, τ)          # which τ? Easy to mix up
S = pulse_spectrum(t, E_t)         # E_t computed elsewhere
```

**OOP solution:** bundle data and behaviour into a single object.

In [3]:
# Preview — by the end of this session we will have:

# pulse = GaussianPulse(tau=100e-15, wavelength=800e-9, energy=50e-6)
# print(pulse)                    # readable repr
# I = pulse.intensity(t)          # data and method travel together
# pulse.tau = -1e-15              # → ValueError: tau must be positive
# chirped = ChirpedPulse(pulse, GDD=500e-30)
# seq = pulse + chirped           # PulseSequence via __add__

print("OOP preview — we will build all of this step by step.")

OOP preview — we will build all of this step by step.


---
## Block 2 — Classes and instances

A **class** is a blueprint. An **instance** is a concrete object created from that blueprint.

```
class GaussianPulse:          ← blueprint
    def __init__(self, ...):  ← called when creating an instance
        self.tau = ...        ← instance attribute

pulse = GaussianPulse(...)    ← instance
```

`self` always refers to the instance being operated on. It is passed automatically — you never call `method(self, ...)` explicitly.

In [4]:
# --------------------------------------------------------------------------
# First version: __init__, instance attributes, instance method
# --------------------------------------------------------------------------

class GaussianPulse:
    """A transform-limited Gaussian pulse."""

    def __init__(self, tau: float, wavelength: float, energy: float) -> None:
        """
        Parameters
        ----------
        tau : float
            1/e field half-width (s).
        wavelength : float
            Central wavelength (m).
        energy : float
            Pulse energy (J).
        """
        self.tau        = tau
        self.wavelength = wavelength
        self.energy     = energy

    def intensity(self, t: np.ndarray) -> np.ndarray:
        """Normalised intensity envelope I(t) = exp(-t²/τ²)."""
        return np.exp(-t**2 / self.tau**2)

    def peak_power(self) -> float:
        """Peak power P₀ = E / (τ√π)."""
        return self.energy / (self.tau * np.sqrt(np.pi))


# Create instances
pulse_800  = GaussianPulse(tau=100e-15, wavelength=800e-9,  energy=50e-6)
pulse_1030 = GaussianPulse(tau=200e-15, wavelength=1030e-9, energy=100e-6)

t = np.linspace(-1e-12, 1e-12, 1024)

print(f"tau       : {pulse_800.tau*1e15:.0f} fs")
print(f"peak power: {pulse_800.peak_power()*1e-6:.2f} MW")
print(f"I(0)      : {pulse_800.intensity(np.array([0.0]))[0]:.4f}")

tau       : 100 fs
peak power: 282.09 MW
I(0)      : 1.0000


In [ ]:
# --------------------------------------------------------------------------
# __repr__ and __str__
# --------------------------------------------------------------------------

class GaussianPulse:
    """A transform-limited Gaussian pulse."""

    def __init__(self, tau: float, wavelength: float, energy: float) -> None:
        self.tau        = tau
        self.wavelength = wavelength
        self.energy     = energy

    def __repr__(self) -> str:
        """Unambiguous representation — used when the object is printed."""
        return (
            f"GaussianPulse("
            f"tau={self.tau*1e15:.1f} fs, "
            f"wavelength={self.wavelength*1e9:.0f} nm, "
            f"energy={self.energy*1e6:.1f} µJ)"
        )

    def __str__(self) -> str:
        """Human-readable description — used by print()."""
        return (
            f"Gaussian pulse @ {self.wavelength*1e9:.0f} nm  "
            f"τ={self.tau*1e15:.0f} fs  E={self.energy*1e6:.1f} µJ  "
            f"P₀={self.peak_power()*1e-6:.2f} MW"
        )

    def intensity(self, t: np.ndarray) -> np.ndarray:
        return np.exp(-t**2 / self.tau**2)

    def peak_power(self) -> float:
        return self.energy / (self.tau * np.sqrt(np.pi))

    def fwhm(self) -> float:
        """Intensity FWHM = 2√(ln2) · τ."""
        return 2 * np.sqrt(np.log(2)) * self.tau


p = GaussianPulse(tau=100e-15, wavelength=800e-9, energy=50e-6)
print(repr(p))   # __repr__
print(str(p))    # __str__
print(p)         # also __str__
print(f"FWHM: {p.fwhm()*1e15:.1f} fs")

GaussianPulse(tau=100.0 fs, wavelength=800 nm, energy=50.0 µJ)
Gaussian pulse @ 800 nm  τ=100 fs  E=50.0 µJ  P₀=282.09 MW
Gaussian pulse @ 800 nm  τ=100 fs  E=50.0 µJ  P₀=282.09 MW
FWHM: 166.5 fs


In [6]:
# --------------------------------------------------------------------------
# Class attributes vs instance attributes
# --------------------------------------------------------------------------

class GaussianPulse:
    # Class attribute: shared by all instances
    c = 3e8   # speed of light (m/s)

    def __init__(self, tau, wavelength, energy):
        # Instance attributes: unique to each instance
        self.tau        = tau
        self.wavelength = wavelength
        self.energy     = energy

    def __repr__(self):
        return (f"GaussianPulse(tau={self.tau*1e15:.1f} fs, "
                f"wavelength={self.wavelength*1e9:.0f} nm, "
                f"energy={self.energy*1e6:.1f} µJ)")

    def frequency(self) -> float:
        """Central frequency ν₀ = c / λ₀."""
        return self.c / self.wavelength

    def intensity(self, t):
        return np.exp(-t**2 / self.tau**2)

    def peak_power(self):
        return self.energy / (self.tau * np.sqrt(np.pi))

    def fwhm(self):
        return 2 * np.sqrt(np.log(2)) * self.tau

    def spectrum(self, t: np.ndarray):
        """Return (freq, PSD) — centred FFT of the intensity envelope."""
        N  = len(t)
        dt = t[1] - t[0]
        E_t  = np.sqrt(self.intensity(t))
        E_f  = np.fft.fftshift(np.fft.fft(E_t))
        freq = np.fft.fftshift(np.fft.fftfreq(N, d=dt))
        return freq, np.abs(E_f)**2


p = GaussianPulse(100e-15, 800e-9, 50e-6)
print(f"ν₀ = {p.frequency()/1e12:.2f} THz")
print(f"GaussianPulse.c = {GaussianPulse.c:.2e} m/s  (class attr)")

ν₀ = 375.00 THz
GaussianPulse.c = 3.00e+08 m/s  (class attr)


⚠️ **Common Pitfall — mutable default arguments in `__init__`**

```python
class PulseSequence:
    # WRONG: the list is created once and shared by all instances
    def __init__(self, pulses=[]):
        self.pulses = pulses

    # RIGHT: use None as sentinel, create list inside __init__
    def __init__(self, pulses=None):
        self.pulses = pulses if pulses is not None else []
```

This is not specific to OOP — it is a Python gotcha for any function with a mutable default argument. With `@dataclass` the `field(default_factory=list)` pattern handles this correctly.

In [7]:
# Demonstrate the pitfall
class Bad:
    def __init__(self, items=[]):   # shared list!
        self.items = items

class Good:
    def __init__(self, items=None):
        self.items = items if items is not None else []

b1, b2 = Bad(), Bad()
b1.items.append('x')
print(f"Bad:  b2.items = {b2.items}")   # ['x'] — contaminated!

g1, g2 = Good(), Good()
g1.items.append('x')
print(f"Good: g2.items = {g2.items}")   # []    — safe

Bad:  b2.items = ['x']
Good: g2.items = []


---
## Block 3 — Properties and validation

`@property` turns a method into an attribute-like accessor.  
The matching `@<name>.setter` runs validation before assignment. If you do not define a `@<name>.setter`, the the attribute assigned with `@property` is read-only (cannot be  changed at run time)

In [ ]:
# --------------------------------------------------------------------------
# @property and @setter — validated GaussianPulse
# --------------------------------------------------------------------------

class GaussianPulse:
    """Transform-limited Gaussian pulse with validated attributes."""

    c = 3e8

    def __init__(self, tau: float, wavelength: float, energy: float) -> None:
        # Use setters for validation from the start
        self.tau        = tau         # → calls self.tau.setter
        self.wavelength = wavelength
        self.energy     = energy

    # --- tau ---
    @property
    def tau(self) -> float:
        """1/e field half-width (s)."""
        return self._tau

    @tau.setter
    def tau(self, value: float) -> None:
        if value <= 0:
            raise ValueError(f"tau must be positive, got {value}")
        self._tau = float(value)

    # --- wavelength ---
    @property
    def wavelength(self) -> float:
        """Central wavelength (m)."""
        return self._wavelength

    @wavelength.setter
    def wavelength(self, value: float) -> None:
        if value <= 0:
            raise ValueError(f"wavelength must be positive, got {value}")
        self._wavelength = float(value)

    # --- energy ---
    @property
    def energy(self) -> float:
        """Pulse energy (J)."""
        return self._energy

    @energy.setter
    def energy(self, value: float) -> None:
        if value < 0:
            raise ValueError(f"energy must be non-negative, got {value}")
        self._energy = float(value)

    # --- computed properties (read-only) ---
    @property
    def frequency(self) -> float:
        """Central frequency ν₀ = c/λ₀ (Hz)."""
        return self.c / self.wavelength

    @property
    def fwhm(self) -> float:
        """Intensity FWHM (s)."""
        return 2 * np.sqrt(np.log(2)) * self.tau

    def __repr__(self) -> str:
        return (f"GaussianPulse(tau={self.tau*1e15:.1f} fs, "
                f"wavelength={self.wavelength*1e9:.0f} nm, "
                f"energy={self.energy*1e6:.1f} µJ)")

    def intensity(self, t: np.ndarray) -> np.ndarray:
        return np.exp(-t**2 / self.tau**2)

    def peak_power(self) -> float:
        return self.energy / (self.tau * np.sqrt(np.pi))

    def spectrum(self, t: np.ndarray):
        N  = len(t); dt = t[1] - t[0]
        E_f  = np.fft.fftshift(np.fft.fft(np.sqrt(self.intensity(t))))
        freq = np.fft.fftshift(np.fft.fftfreq(N, d=dt))
        return freq, np.abs(E_f)**2


p = GaussianPulse(tau=100e-15, wavelength=800e-9, energy=50e-6)
print(p)
print(f"FWHM     : {p.fwhm*1e15:.1f} fs")   # property, no parentheses
print(f"Frequency: {p.frequency/1e12:.2f} THz")

# Validation
try:
    p.tau = -1e-15
except ValueError as e:
    print(f"\nCaught: {e}")

# Update is allowed with valid value
p.tau = 200e-15
print(f"tau updated: {p.tau*1e15:.0f} fs")

In [ ]:
# ⚡ Try It — add a read-only 'bandwidth' property
#
# For a transform-limited Gaussian pulse:
#   Δν_FWHM = 2ln(2) / (π · τ_FWHM)  =  2ln(2) / (π · 2√(ln2) · τ)
#           = √(ln2) / (π · τ)
#
# Add the property to GaussianPulse and verify:
#   p.bandwidth * p.fwhm ≈ 0.4413  (time-bandwidth product)

# YOUR CODE HERE

# Solution (hidden)
# @property
# def bandwidth(self) -> float:
#     """Spectral bandwidth Δν FWHM (Hz)."""
#     return np.sqrt(np.log(2)) / (np.pi * self.tau)
#
# p = GaussianPulse(100e-15, 800e-9, 50e-6)
# print(f"TBP = {p.bandwidth * p.fwhm:.4f}  (expected 0.4413)")

---
## Block 4 — Special methods (dunder)

Dunder methods (`__method__`) let you define how objects respond to Python operators and built-in functions.

In [ ]:
# --------------------------------------------------------------------------
# PulseSequence — container with dunder methods
# --------------------------------------------------------------------------

class PulseSequence:
    """An ordered collection of GaussianPulse objects."""

    def __init__(self, pulses=None) -> None:
        self._pulses = list(pulses) if pulses is not None else []

    # Container protocol
    def __len__(self) -> int:
        return len(self._pulses)

    def __getitem__(self, idx):
        return self._pulses[idx]

    def __iter__(self):
        return iter(self._pulses)

    def __contains__(self, item) -> bool:
        return item in self._pulses

    # Arithmetic
    def __add__(self, other):
        """Concatenate two sequences or append a single pulse."""
        if isinstance(other, PulseSequence):
            return PulseSequence(self._pulses + other._pulses)
        if isinstance(other, GaussianPulse):
            return PulseSequence(self._pulses + [other])
        return NotImplemented

    def __repr__(self) -> str:
        return f"PulseSequence({len(self)} pulses)"

    # Physics methods
    def total_energy(self) -> float:
        """Sum of all pulse energies (J)."""
        return sum(p.energy for p in self._pulses)

    def sort_by_wavelength(self) -> 'PulseSequence':
        """Return a new sequence sorted by central wavelength."""
        return PulseSequence(sorted(self._pulses, key=lambda p: p.wavelength))


p1 = GaussianPulse(100e-15, 800e-9,  50e-6)
p2 = GaussianPulse(200e-15, 1030e-9, 100e-6)
p3 = GaussianPulse( 50e-15, 515e-9,  10e-6)

seq = PulseSequence([p1, p2])
print(seq)                          # __repr__
print(f"len : {len(seq)}")          # __len__
print(f"[0] : {seq[0]}")            # __getitem__

# Iteration
for pulse in seq:                   # __iter__
    print(f"  {pulse.wavelength*1e9:.0f} nm  τ={pulse.tau*1e15:.0f} fs")

# Concatenation
seq2 = seq + p3                     # __add__
print(f"\nAfter + p3: {seq2}")
print(f"Total energy: {seq2.total_energy()*1e6:.1f} µJ")

# Sorted
sorted_seq = seq2.sort_by_wavelength()
for p in sorted_seq:
    print(f"  {p.wavelength*1e9:.0f} nm")

In [ ]:
# --------------------------------------------------------------------------
# __eq__ and __lt__ — comparison operators
# --------------------------------------------------------------------------

class GaussianPulse(GaussianPulse):   # extend the existing class

    def __eq__(self, other) -> bool:
        if not isinstance(other, GaussianPulse):
            return NotImplemented
        return (np.isclose(self.tau, other.tau) and
                np.isclose(self.wavelength, other.wavelength) and
                np.isclose(self.energy, other.energy))

    def __lt__(self, other) -> bool:
        """Compare by energy."""
        if not isinstance(other, GaussianPulse):
            return NotImplemented
        return self.energy < other.energy


pa = GaussianPulse(100e-15, 800e-9, 50e-6)
pb = GaussianPulse(100e-15, 800e-9, 50e-6)
pc = GaussianPulse(100e-15, 800e-9, 80e-6)

print(f"pa == pb : {pa == pb}")   # True
print(f"pa == pc : {pa == pc}")   # False
print(f"pa < pc  : {pa < pc}")    # True (50 < 80 µJ)
print(f"sorted   : {[p.energy*1e6 for p in sorted([pc, pa, pb])]}")   # uses __lt__

---
## Block 5 — Inheritance

Inheritance lets a child class **reuse** parent code and **override** specific behaviour.  
`super()` calls the parent implementation — essential in `__init__` to avoid duplicating attribute setup.

In [8]:
# --------------------------------------------------------------------------
# ChirpedPulse inherits from GaussianPulse
# --------------------------------------------------------------------------

class ChirpedPulse(GaussianPulse):
    """
    A Gaussian pulse with group delay dispersion (GDD).

    The chirped field in frequency domain:
        E_chirped(ω) = E_TL(ω) · exp(i · GDD/2 · ω²)
    """

    def __init__(
        self,
        tau: float,
        wavelength: float,
        energy: float,
        gdd: float,
    ) -> None:
        super().__init__(tau, wavelength, energy)   # delegate to parent
        self.gdd = gdd   # group delay dispersion (s²)

    @property
    def gdd(self) -> float:
        return self._gdd

    @gdd.setter
    def gdd(self, value: float) -> None:
        self._gdd = float(value)   # GDD can be negative

    @property
    def tau_chirped(self) -> float:
        """Chirped pulse duration: τ_c = τ √(1 + (GDD/τ²)²)."""
        return self.tau * np.sqrt(1 + (self.gdd / self.tau**2)**2)

    def __repr__(self) -> str:
        return (
            f"ChirpedPulse(tau={self.tau*1e15:.1f} fs, "
            f"wavelength={self.wavelength*1e9:.0f} nm, "
            f"energy={self.energy*1e6:.1f} µJ, "
            f"gdd={self.gdd*1e30:.0f} fs²)"
        )

    def intensity(self, t: np.ndarray) -> np.ndarray:
        """Override: chirped intensity has broader envelope."""
        return np.exp(-t**2 / self.tau_chirped**2)

    def spectrum(self, t: np.ndarray):
        """Override: apply GDD phase in frequency domain."""
        N  = len(t); dt = t[1] - t[0]
        E_TL = np.sqrt(np.exp(-t**2 / self.tau**2))
        freq  = np.fft.fftshift(np.fft.fftfreq(N, d=dt))
        ω     = 2 * np.pi * freq
        E_f   = np.fft.fftshift(np.fft.fft(E_TL))
        E_f  *= np.exp(0.5j * self.gdd * ω**2)
        return freq, np.abs(E_f)**2


p_TL  = GaussianPulse(tau=100e-15, wavelength=800e-9, energy=50e-6)
p_ch  = ChirpedPulse( tau=100e-15, wavelength=800e-9, energy=50e-6, gdd=500e-30)

print(p_TL)
print(p_ch)
print(f"\nChirped duration: {p_ch.tau_chirped*1e15:.1f} fs")
print(f"isinstance(p_ch, GaussianPulse) : {isinstance(p_ch, GaussianPulse)}")
print(f"isinstance(p_TL, ChirpedPulse)  : {isinstance(p_TL, ChirpedPulse)}")

GaussianPulse(tau=100.0 fs, wavelength=800 nm, energy=50.0 µJ)
ChirpedPulse(tau=100.0 fs, wavelength=800 nm, energy=50.0 µJ, gdd=500 fs²)

Chirped duration: 100.1 fs
isinstance(p_ch, GaussianPulse) : True
isinstance(p_TL, ChirpedPulse)  : False


In [ ]:
# --------------------------------------------------------------------------
# Polymorphism — same interface, different behaviour
# --------------------------------------------------------------------------

t = np.linspace(-2e-12, 2e-12, 2048)

pulses = [
    GaussianPulse(100e-15, 800e-9, 50e-6),
    ChirpedPulse( 100e-15, 800e-9, 50e-6, gdd=500e-30),
    ChirpedPulse( 100e-15, 800e-9, 50e-6, gdd=2000e-30),
]

fig, ax = plt.subplots(figsize=(7, 3.5))
for pulse in pulses:              # same call for all types — polymorphism
    ax.plot(t*1e15, pulse.intensity(t), lw=2, label=repr(pulse)[:30])

ax.set_xlabel('Time (fs)')
ax.set_ylabel('Intensity (norm.)')
ax.set_title('GaussianPulse vs ChirpedPulse — polymorphism')
ax.set_xlim(-1500, 1500)
ax.legend(fontsize=7)
fig.tight_layout()
plt.show()

# wihtout polymorphism, you would write:

# for pulse in pulses:
#    if isinstance(pulse, GaussianPulse):
#        ax.plot(t, gaussian_intensity(t, pulse.tau))
#    elif isinstance(pulse, ChirpedPulse):
#        ax.plot(t, chirped_intensity(t, pulse.tau, pulse.gdd))

# This breaks every time you add a new pulse type. With polymorphism you add a new class 
# with .intensity(t) and the loop works automatically — no changes needed.
# This is why polymorphism is one of the core ideas in OOP: 
# it lets you write code that works on categories of objects rather than specific types.

In [ ]:
# --------------------------------------------------------------------------
# OpticalElement hierarchy — shallow (option A)
# --------------------------------------------------------------------------

class OpticalElement:
    """Abstract base for ABCD optical elements."""

    def matrix(self) -> np.ndarray:
        """Return the 2×2 ABCD ray transfer matrix."""
        raise NotImplementedError("Subclasses must implement matrix()")

    def __matmul__(self, other):
        """Compose two elements: (self @ other).matrix() = self.matrix() @ other.matrix()."""
        if isinstance(other, OpticalElement):
            M = self.matrix() @ other.matrix()
            return _MatrixElement(M)
        return NotImplemented

    def propagate(self, ray: np.ndarray) -> np.ndarray:
        """Apply this element to a ray [y, θ]."""
        return self.matrix() @ ray

    def __repr__(self) -> str:
        return f"{self.__class__.__name__}()"


class _MatrixElement(OpticalElement):
    """Internal: holds a precomputed matrix (result of composition)."""
    def __init__(self, M): self._M = M
    def matrix(self): return self._M


class FreeSpace(OpticalElement):
    """Free-space propagation over distance d."""

    def __init__(self, d: float) -> None:
        if d < 0:
            raise ValueError(f"Distance must be non-negative, got {d}")
        self.d = d

    def matrix(self) -> np.ndarray:
        return np.array([[1.0, self.d], [0.0, 1.0]])

    def __repr__(self) -> str:
        return f"FreeSpace(d={self.d*1e3:.1f} mm)"


class ThinLens(OpticalElement):
    """Thin lens with focal length f."""

    def __init__(self, f: float) -> None:
        if f == 0:
            raise ValueError("Focal length cannot be zero")
        self.f = f

    def matrix(self) -> np.ndarray:
        return np.array([[1.0, 0.0], [-1.0/self.f, 1.0]])

    def __repr__(self) -> str:
        return f"ThinLens(f={self.f*1e3:.1f} mm)"


class Mirror(OpticalElement):
    """Curved mirror with radius of curvature R (R>0: concave)."""

    def __init__(self, R: float) -> None:
        self.R = R

    def matrix(self) -> np.ndarray:
        return np.array([[1.0, 0.0], [-2.0/self.R, 1.0]])

    def __repr__(self) -> str:
        return f"Mirror(R={self.R*1e3:.1f} mm)"


# Build a system using @
system = FreeSpace(0.5) @ ThinLens(0.25) @ FreeSpace(0.5)
ray_in = np.array([5e-3, 0.0])
ray_out = system.propagate(ray_in)

print(FreeSpace(0.5))
print(ThinLens(0.25))
print(Mirror(0.5))
print(f"\nSystem matrix:\n{system.matrix().round(4)}")
print(f"Ray in : y={ray_in[0]*1e3:.1f} mm, θ={ray_in[1]*1e3:.1f} mrad")
print(f"Ray out: y={ray_out[0]*1e6:.2f} µm, θ={ray_out[1]*1e3:.2f} mrad")

---
## Block 6 — Dataclasses and NamedTuple

For data containers that do not need complex methods, Python offers two lightweight alternatives to full classes.

In [ ]:
# --------------------------------------------------------------------------
# @dataclass — auto-generates __init__, __repr__, __eq__
# --------------------------------------------------------------------------

from dataclasses import dataclass, field

@dataclass
class BeamParameters:
    """
    Spatial parameters of a Gaussian beam at a given plane.

    All lengths in metres.
    """
    wavelength : float           # m
    w0         : float           # beam waist radius (m)
    z          : float = 0.0    # position along optical axis (m)
    M2         : float = 1.0    # beam quality factor

    # Computed fields — not constructor arguments
    def __post_init__(self):
        if self.w0 <= 0:
            raise ValueError(f"w0 must be positive, got {self.w0}")
        if self.M2 < 1:
            raise ValueError(f"M² must be ≥ 1, got {self.M2}")

    @property
    def rayleigh_range(self) -> float:
        """z_R = π w₀² / (M² λ)."""
        return np.pi * self.w0**2 / (self.M2 * self.wavelength)

    @property
    def w_at_z(self) -> float:
        """Beam radius at current z."""
        return self.w0 * np.sqrt(1 + (self.z / self.rayleigh_range)**2)


beam = BeamParameters(wavelength=800e-9, w0=500e-6, z=0.1, M2=1.2)
print(beam)                      # auto-generated __repr__
print(f"zR   = {beam.rayleigh_range*100:.2f} cm")
print(f"w(z) = {beam.w_at_z*1e6:.1f} µm")

# Auto-generated __eq__
beam2 = BeamParameters(wavelength=800e-9, w0=500e-6, z=0.1, M2=1.2)
print(f"beam == beam2 : {beam == beam2}")

# Validation
try:
    BeamParameters(wavelength=800e-9, w0=-1e-6)
except ValueError as e:
    print(f"Caught: {e}")

In [ ]:
# --------------------------------------------------------------------------
# frozen=True — immutable dataclass
# --------------------------------------------------------------------------

@dataclass(frozen=True)
class LaserLine:
    """
    Immutable record of a laser emission line.
    frozen=True: instances are hashable and can be used as dict keys.
    """
    wavelength_nm : float
    gain_medium   : str
    peak_power_W  : float


ti_sapphire = LaserLine(800.0,  'Ti:Sapphire', 1e9)
yb_fiber    = LaserLine(1030.0, 'Yb:fiber',    1e6)

print(ti_sapphire)
print(f"Hashable: {hash(ti_sapphire)}")

# Immutability
try:
    ti_sapphire.wavelength_nm = 810.0
except Exception as e:
    print(f"Caught: {e}")

# Use as dict key
pulse_count = {ti_sapphire: 1000, yb_fiber: 5000}
print(f"Ti:Sa shots: {pulse_count[ti_sapphire]}")

In [ ]:
# --------------------------------------------------------------------------
# NamedTuple — immutable, unpackable, memory-efficient
# --------------------------------------------------------------------------

from typing import NamedTuple

class MeasurementPoint(NamedTuple):
    """
    A single data point from a beam profiler measurement.
    Immutable and unpackable like a regular tuple.
    """
    x_mm    : float   # horizontal position (mm)
    y_mm    : float   # vertical position (mm)
    fluence : float   # J/cm²


pt = MeasurementPoint(x_mm=0.5, y_mm=-0.1, fluence=1.23)
print(pt)

# Unpacking — works like a regular tuple
x, y, f = pt
print(f"x={x} mm, y={y} mm, fluence={f} J/cm²")

# Named access
print(f"pt.fluence = {pt.fluence}")

# Build a dataset as a list of NamedTuples
rng  = np.random.default_rng(1)
data = [
    MeasurementPoint(x, y, float(np.exp(-2*(x**2+y**2)/0.5**2)))
    for x in np.linspace(-1, 1, 5)
    for y in np.linspace(-1, 1, 5)
]
print(f"\nDataset: {len(data)} points")
print(f"Max fluence: {max(pt.fluence for pt in data):.3f} J/cm²")

💡 **Pro Tip — when to use which**

| Tool | Use when |
|------|----------|
| Full `class` | Need complex methods, inheritance, or mutable state with validation |
| `@dataclass` | Primarily a data container; want auto `__init__`/`__repr__`/`__eq__` |
| `@dataclass(frozen=True)` | Immutable parameter set; Same as above but fields cannot be changed after creation|
| `NamedTuple` | Lightweight record; need tuple unpacking; read-only |
| `dict` | One-off, no type safety needed |


---
## Block 7 — Integrating with `shared/`

The classes developed today go into two new files in `shared/`, **parallel** to the existing functional code.

In [ ]:
# --------------------------------------------------------------------------
# shared/pulse_classes.py — what to commit
# --------------------------------------------------------------------------

# The file should contain:
#
#   class GaussianPulse:
#       - __init__(tau, wavelength, energy)
#       - @property tau / wavelength / energy  (with validation)
#       - @property frequency / fwhm / bandwidth
#       - intensity(t), spectrum(t), peak_power()
#       - __repr__, __eq__, __lt__
#
#   class ChirpedPulse(GaussianPulse):
#       - __init__(tau, wavelength, energy, gdd)
#       - @property gdd, tau_chirped
#       - intensity(t)  [overrides]
#       - spectrum(t)   [overrides]
#       - __repr__
#
# These classes are PARALLEL to the functions in shared/pulses.py.
# Both can coexist — functions for quick scripts, classes for pipelines.

# Quick self-test (run this after creating the file)
# from shared.pulse_classes import GaussianPulse, ChirpedPulse

t_test = np.linspace(-1e-12, 1e-12, 512)

p_test = GaussianPulse(100e-15, 800e-9, 50e-6)
assert p_test.intensity(np.array([0.0]))[0] == 1.0
assert p_test.fwhm > 0
assert np.isclose(p_test.fwhm * p_test.bandwidth * 2 * np.pi, 4 * np.log(2), rtol=0.01)

cp_test = ChirpedPulse(100e-15, 800e-9, 50e-6, gdd=500e-30)
assert cp_test.tau_chirped > cp_test.tau
_, S_TL = p_test.spectrum(t_test)
_, S_ch = cp_test.spectrum(t_test)
assert np.allclose(S_TL / S_TL.max(), S_ch / S_ch.max(), atol=1e-6), \
    "GDD must not change the spectrum shape"

print("shared/pulse_classes.py self-test passed ✓")

In [ ]:
# --------------------------------------------------------------------------
# shared/beam_classes.py — what to commit
# --------------------------------------------------------------------------

# Contains:
#   class OpticalElement  (abstract base)
#   class FreeSpace(OpticalElement)
#   class ThinLens(OpticalElement)
#   class Mirror(OpticalElement)
#   @dataclass BeamParameters

# Self-test
fs = FreeSpace(0.5)
tl = ThinLens(0.25)
sys = fs @ tl @ fs
assert isinstance(sys, OpticalElement)
M = sys.matrix()
assert M.shape == (2, 2)

bp = BeamParameters(wavelength=800e-9, w0=500e-6)
assert bp.rayleigh_range > 0
assert np.isclose(bp.w_at_z, bp.w0)   # z=0 by default

print("shared/beam_classes.py self-test passed ✓")
print()
print("Git commands to commit:")
print("  git add shared/pulse_classes.py shared/beam_classes.py")
print("  git commit -m 'S07: add OOP pulse and beam classes to shared/'")

---
## Summary

| Concept | Syntax |
|---------|--------|
| Define class | `class MyClass:` |
| Constructor | `def __init__(self, ...):` |
| Instance method | `def method(self, ...):` |
| Readable repr | `def __repr__(self):` |
| Validated attribute | `@property` + `@name.setter` |
| Computed read-only | `@property` without setter |
| Container protocol | `__len__`, `__getitem__`, `__iter__`, `__contains__` |
| Arithmetic | `__add__`, `__mul__`, `__matmul__` |
| Comparison | `__eq__`, `__lt__` |
| Inherit | `class Child(Parent):` |
| Call parent | `super().__init__(...)` |
| Override | redefine method in child |
| Check type | `isinstance(obj, Class)` |
| Data container | `@dataclass` / `@dataclass(frozen=True)` |
| Lightweight record | `class X(NamedTuple):` |
| Mutable default | `field(default_factory=list)` in dataclass |

**`shared/` additions (parallel, do not touch existing files):**
- `shared/pulse_classes.py` ← `GaussianPulse`, `ChirpedPulse`
- `shared/beam_classes.py` ← `OpticalElement`, `FreeSpace`, `ThinLens`, `Mirror`, `BeamParameters`

**Next: S08 — Testing, Type Hints & Defensive Programming**